In [ ]:
#!pip install contractions


In [ ]:
import nltk
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import contractions
import pandas as pd
import re, string


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')


In [ ]:
df = pd.read_csv('Spam_SMS.csv')
df.head()


# Data Exploratory


In [ ]:
print(df.shape) # Number of rows and columns in a data frame
df.info()# Brief information about the data frame


In [ ]:
# Distribution of categories
print("\nDistribution of 'Class' column:")
print(df['Class'].value_counts())

# Indicates data imbalance.


In [ ]:
# Check for unique and missing values
print(df["Class"].unique())
print(df.isnull().sum())


In [ ]:
# Show statistics
df['Message_Length'] = df['Message'].apply(len)
df[['Message_Length']].describe()


In [ ]:
# Remove duplicates
print(df.duplicated().sum())
df.drop_duplicates(inplace=True)


In [ ]:
# We convert them into numbers so the model can handle them.
df["Class"] = df["Class"].map({'ham': 0, 'spam': 1})


In [ ]:
# Expand abbreviations using the contractions library
df["Expanded_Message"] = df["Message"].apply(lambda text: contractions.fix(text) if isinstance(text, str) else "")


In [ ]:
# stop words
stop_words = set(stopwords.words('english'))


# **Text Preprocessing**


In [ ]:
# Word processing
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.strip()
    text = ' '.join(text.split())
    # Stop Words
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

df["Cleaned_Message"] = df["Expanded_Message"].apply(clean_text)

# Lemmatization
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def apply_lemmatization(text):
    lemmatized_text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    return lemmatized_text

df["Lemmatized_Message"] = df["Cleaned_Message"].apply(apply_lemmatization)
df.head()


In [ ]:
from sklearn.model_selection import train_test_split
# Splitting the data into two sets: training and testing
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df['Lemmatized_Message'],df['Class'],test_size=0.2,random_state=42,stratify=df['Class'])
# stratify=df['Class'] >>> Maintaining a balanced distribution of classes in both sets


# **Text Representation**


a. CountVectorizer (Bag of Words)


In [ ]:
# a. CountVectorizer (Bag of Words)
count_vectorizer = CountVectorizer()
X_train_bow = count_vectorizer.fit_transform(X_train_raw)  # "fit_transform" On training data
X_test_bow = count_vectorizer.transform(X_test_raw)        # "transform" On test data


b. TfidfVectorizer (TF-IDF)


In [ ]:
# b. TfidfVectorizer (TF-IDF)
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)  # "fit_transform" On training data
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)        # "transform" On test data


c. Compare the BoW and TF-IDF matrices (e.g., shape, sparsity)


- Both the BoW and TF-IDF matrices have the same shape and a high sparsity of approximately 0.99, indicating that they are both very sparse.

- The BoW matrix uses integer counts, while the TF-IDF matrix uses floating-point weights.

- This similarity in sparsity indicates that both representations represent a similar proportion of the vocabulary in the documents.


In [ ]:
print("BoW Matrix:")
print("Shape:", X_train_bow.shape)
print("Data Type:", X_train_bow.dtype)
print("Non-zero elements:", X_train_bow.nnz)
print("Sparsity:", 1 - (X_train_bow.nnz / (X_train_bow.shape[0] * X_train_bow.shape[1])))

print("\nTF-IDF Matrix:")
print("Shape:", X_train_tfidf.shape)
print("Data Type:", X_train_tfidf.dtype)
print("Non-zero elements:", X_train_tfidf.nnz)
print("Sparsity:", 1 - (X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1])))


In [ ]:
# Print dimensions for verification
print("X_train_bow:...", X_train_bow.shape)
print("X_test_bow:....", X_test_bow.shape)
print("X_train_tfidf:.", X_train_tfidf.shape)
print("X_test_tfidf:..", X_test_tfidf.shape)


# **Data Visualization**


a. Plot the top 10 most frequent words in spam vs. ham messages


In [ ]:
import matplotlib.pyplot as plt

# Separate data into Spam and Ham
spam_df = df[df['Class'] == 1]
ham_df = df[df['Class'] == 0]

# Counting word frequency using CountVectorizer
def get_top_words(text_series, n=10):

    vec = CountVectorizer().fit(text_series)
    bag_of_words = vec.transform(text_series)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return pd.DataFrame(words_freq[:n], columns=['word', 'count'])

# Count the most frequently occurring words in each category.
top_spam_words = get_top_words(spam_df['Cleaned_Message'], n=10)
top_ham_words = get_top_words(ham_df['Cleaned_Message'], n=10)

# Drawing duplicates
fig, axes = plt.subplots(1, 2, figsize=(16, 6))  # One row, two columns (two charts)

# Spam
sns.barplot(x='count', y='word', data=top_spam_words, ax=axes[0], color='blue', orient='h')
axes[0].set_title('Top 10 Most Frequent Words in Spam Messages')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Word')

# Ham
sns.barplot(x='count', y='word', data=top_ham_words, ax=axes[1], color='green', orient='h')
axes[1].set_title('Top 10 Most Frequent Words in Ham Messages')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Word')

plt.tight_layout()  # Prevent overlapping addresses
plt.show()


b. Create a word cloud for spam and ham messages separately


In [ ]:
# Prepare and customize stop words
stopwords = set(STOPWORDS)
custom_stopwords = ['u', 'ur', '2', '4', 'n', 'im', 'dont', 'ltgt']
stopwords.update(custom_stopwords)

# Plot Word Cloud
def plot_wordcloud(text_series, title, ax):

    wordcloud = WordCloud(width=800, height=400, background_color='white', stopwords=stopwords, min_font_size=10).generate(' '.join(text_series))

    ax.imshow(wordcloud, interpolation='bilinear')
    ax.axis("off")
    ax.set_title(title)


# Create Subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

# Generate and Plot Word Clouds
plot_wordcloud(spam_df['Lemmatized_Message'], "Word Cloud for Spam Messages", axes[0])
plot_wordcloud(ham_df['Lemmatized_Message'], "Word Cloud for Ham Messages", axes[1])

plt.tight_layout()
plt.show()


# **Spam Classification using**


1. Train a Naïve Bayes classifier on BoW and TF-IDF representations.


In [ ]:
# BoW
model_bow = MultinomialNB()
model_bow.fit(X_train_bow, y_train)


In [ ]:
# TF-IDF
model_tfidf = MultinomialNB()
model_tfidf.fit(X_train_tfidf, y_train)


2. Train a Logistic Regression classifier on BoW and TF-IDF representations.**
- Increase max_iter to avoid non-convergence warnings


In [ ]:
# BoW
model_bow_lr = LogisticRegression(max_iter=1000)
model_bow_lr.fit(X_train_bow, y_train)


In [ ]:
# TF-IDF
model_tfidf_lr = LogisticRegression(max_iter=1000)
model_tfidf_lr.fit(X_train_tfidf, y_train)


3. Train a Support Vector Machine (SVM) on BoW and TF-IDF representations
(optional)


In [ ]:
# BoW
model_bow_svm = SVC(kernel='linear', probability=True)
model_bow_svm.fit(X_train_bow, y_train)


In [ ]:
# TF-IDF
model_tfidf_svm = SVC(kernel='linear', probability=True)
model_tfidf_svm.fit(X_train_tfidf, y_train)


# **Model Evaluation and Comparison**


**Evaluate each model using:**

a. Accuracy, Precision, Recall and F1-score

b. Confusion matrices for each model

c. ROC-AUC curve comparison (optional)


In [ ]:
# Imports
# from sklearn.metrics import accuracy_score,confusion_matrix ,roc_auc_score, RocCurveDisplay
# from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_curve, auc
# import matplotlib.pyplot as plt


**This model achieved high accuracy and good performance in spam recognition, with a good balance between precision and recall.**


In [ ]:
# Evaluate Naive Bayes | BoW
y_pred_bow = model_bow.predict(X_test_bow)  # prediction

print("Naive Bayes with BoW:")
print("Accuracy:", accuracy_score(y_test, y_pred_bow))
print(classification_report(y_test, y_pred_bow))

cm_bow = confusion_matrix(y_test, y_pred_bow)
disp_bow = ConfusionMatrixDisplay(confusion_matrix=cm_bow, display_labels=['ham', 'spam'])
disp_bow.plot()
plt.title('Confusion Matrix - Naive Bayes (BoW)')
plt.show()

y_prob_bow = model_bow.predict_proba(X_test_bow)[:, 1]
fpr_bow, tpr_bow, thresholds_bow = roc_curve(y_test, y_prob_bow)
roc_auc_bow = auc(fpr_bow, tpr_bow)

display_bow = RocCurveDisplay(fpr=fpr_bow, tpr=tpr_bow, roc_auc=roc_auc_bow, estimator_name="Naive Bayes (BoW)")
display_bow.plot()
plt.title("ROC-AUC Curve - Naive Bayes (BoW)")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.show()


**Although it has a good AUC, this model suffers from a significant drop in spam recall, which means it misses many real spam messages.**


In [ ]:
# Evaluate Naive Bayes | TF-IDF
y_pred_tfidf = model_tfidf.predict(X_test_tfidf)

print("\nNaive Bayes with TF-IDF:")
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print(classification_report(y_test, y_pred_tfidf))

cm_tfidf = confusion_matrix(y_test, y_pred_tfidf)
disp_tfidf = ConfusionMatrixDisplay(confusion_matrix=cm_tfidf, display_labels=['ham', 'spam'])
disp_tfidf.plot()
plt.title('Confusion Matrix - Naive Bayes (TF-IDF)')
plt.show()

y_prob_tfidf = model_tfidf.predict_proba(X_test_tfidf)[:, 1]
fpr_tfidf, tpr_tfidf, thresholds_tfidf = roc_curve(y_test, y_prob_tfidf)
roc_auc_tfidf = auc(fpr_tfidf, tpr_tfidf)

display_tfidf = RocCurveDisplay(fpr=fpr_tfidf, tpr=tpr_tfidf, roc_auc=roc_auc_tfidf, estimator_name="Naive Bayes (TF-IDF)")
display_tfidf.plot()
plt.title("ROC-AUC Curve - Naive Bayes (TF-IDF)")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.show()


**This model performs very competitively with Naive Bayes on BoW, with similar accuracy and a higher AUC, but may be slightly less able to recall all spam messages.**


In [ ]:
# Evaluate Logistic Regression | BoW
y_pred_bow_lr = model_bow_lr.predict(X_test_bow)

print("\nLogistic Regression with BoW:")
print("Accuracy:", accuracy_score(y_test, y_pred_bow_lr))
print(classification_report(y_test, y_pred_bow_lr))

cm_bow_lr = confusion_matrix(y_test, y_pred_bow_lr)
disp_bow_lr = ConfusionMatrixDisplay(confusion_matrix=cm_bow_lr, display_labels=['ham', 'spam'])
disp_bow_lr.plot()
plt.title('Confusion Matrix - Logistic Regression (BoW)')
plt.show()

y_prob_bow_lr = model_bow_lr.predict_proba(X_test_bow)[:, 1]
fpr_bow_lr, tpr_bow_lr, thresholds_bow_lr = roc_curve(y_test, y_prob_bow_lr)
roc_auc_bow_lr = auc(fpr_bow_lr, tpr_bow_lr)

display_bow_lr = RocCurveDisplay(fpr=fpr_bow_lr, tpr=tpr_bow_lr, roc_auc=roc_auc_bow_lr, estimator_name="Logistic Regression (BoW)")
display_bow_lr.plot()
plt.title("ROC-AUC Curve - Logistic Regression (BoW)")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.show()


**Similar to Naive Bayes with TF-IDF, this model has excellent AUC but suffers from a significant decrease in recall for spam messages.**


In [ ]:
# Evaluate Logistic Regression | TF-IDF
y_pred_tfidf_lr = model_tfidf_lr.predict(X_test_tfidf)

print("\nLogistic Regression with TF-IDF:")
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf_lr))
print(classification_report(y_test, y_pred_tfidf_lr))

cm_tfidf_lr = confusion_matrix(y_test, y_pred_tfidf_lr)
disp_tfidf_lr = ConfusionMatrixDisplay(confusion_matrix=cm_tfidf_lr, display_labels=['ham', 'spam'])
disp_tfidf_lr.plot()
plt.title('Confusion Matrix - Logistic Regression (TF-IDF)')
plt.show()

y_prob_tfidf_lr = model_tfidf_lr.predict_proba(X_test_tfidf)[:, 1]
fpr_tfidf_lr, tpr_tfidf_lr, thresholds_tfidf_lr = roc_curve(y_test, y_prob_tfidf_lr)
roc_auc_tfidf_lr = auc(fpr_tfidf_lr, tpr_tfidf_lr)

display_tfidf_lr = RocCurveDisplay(fpr=fpr_tfidf_lr, tpr=tpr_tfidf_lr, roc_auc=roc_auc_tfidf_lr, estimator_name="Logistic Regression (TF-IDF)")
display_tfidf_lr.plot()
plt.title("ROC-AUC Curve - Logistic Regression (TF-IDF)")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.show()


**This model offers excellent performance, competing with Naive Bayes on BoW as one of the best models, with high accuracy and a good F1-score for spam.**


In [ ]:
# Evaluate SVM | BoW
y_pred_bow_svm = model_bow_svm.predict(X_test_bow)

print("\nSVM with BoW:")
print("Accuracy:", accuracy_score(y_test, y_pred_bow_svm))
print(classification_report(y_test, y_pred_bow_svm))

cm_bow_svm = confusion_matrix(y_test, y_pred_bow_svm)
disp_bow_svm = ConfusionMatrixDisplay(confusion_matrix=cm_bow_svm, display_labels=['ham', 'spam'])
disp_bow_svm.plot()
plt.title('Confusion Matrix - SVM (BoW)')
plt.show()

y_prob_bow_svm = model_bow_svm.predict_proba(X_test_bow)[:, 1]
fpr_bow_svm, tpr_bow_svm, thresholds_bow_svm = roc_curve(y_test, y_prob_bow_svm)
roc_auc_bow_svm = auc(fpr_bow_svm, tpr_bow_svm)

display_bow_svm = RocCurveDisplay(fpr=fpr_bow_svm, tpr=tpr_bow_svm, roc_auc=roc_auc_bow_svm, estimator_name="SVM (BoW)")
display_bow_svm.plot()
plt.title("ROC-AUC Curve - SVM (BoW)")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.show()


**This model appears to be the best performer overall, achieving the highest accuracy and highest F1-score for spam (combined with other models), as well as an excellent AUC value.**


In [ ]:
# Evaluate SVM | TF-IDF
y_pred_tfidf_svm = model_tfidf_svm.predict(X_test_tfidf)

print("\nSVM with TF-IDF:")
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf_svm))
print(classification_report(y_test, y_pred_tfidf_svm))

cm_tfidf_svm = confusion_matrix(y_test, y_pred_tfidf_svm)
disp_tfidf_svm = ConfusionMatrixDisplay(confusion_matrix=cm_tfidf_svm, display_labels=['ham', 'spam'])
disp_tfidf_svm.plot()
plt.title('Confusion Matrix - SVM (TF-IDF)')
plt.show()

y_prob_tfidf_svm = model_tfidf_svm.predict_proba(X_test_tfidf)[:, 1]
fpr_tfidf_svm, tpr_tfidf_svm, thresholds_tfidf_svm = roc_curve(y_test, y_prob_tfidf_svm)
roc_auc_tfidf_svm = auc(fpr_tfidf_svm, tpr_tfidf_svm)

display_tfidf_svm = RocCurveDisplay(fpr=fpr_tfidf_svm, tpr=tpr_tfidf_svm, roc_auc=roc_auc_tfidf_svm, estimator_name="SVM (TF-IDF)")
display_tfidf_svm.plot()
plt.title("ROC-AUC Curve - SVM (TF-IDF)")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.show()


# **Write one paragraph discussing:**


a. Which model performed best and why?


The SVM model trained on TF-IDF features appears to be the best performer . It achieved the highest overall accuracy of 97.67%, as well as the highest F1-score for spam at 0.90, and an excellent AUC of 0.98. Although the recall for spam (0.84) was not the highest , the overall balance of high accuracy, good spam recall, and strong ability to discriminate between the two classes (as evidenced by the AUC) makes it the best model among the evaluated models.


b. How does BoW compare to TF-IDF?


Overall, when comparing the performance of models trained using Bag-of-Words (BoW) with those using TF-IDF, we observed that models trained using BoW performed better or similarly in terms of accuracy and, more importantly, in correctly identifying spam messages (higher Recall and F1-score for spam), especially for the Naive Bayes and Logistic Regression models. Although TF-IDF sometimes resulted in slightly better AUC values ​​(indicating overall better discriminative ability across different thresholds), performance at a specific classification threshold appeared to favor BoW in many cases. However, for the SVM model, TF-IDF features led to the best overall performance across most metrics.


c. Challenges in spam detection using ML?


Detecting spam using machine learning faces several challenges, most notably the ever-changing nature of spammers' tactics, who are constantly innovating new ways to bypass filters, such as using new keywords, obfuscation techniques, and changing message structures. Additionally, spam datasets are often highly imbalanced, with legitimate messages far outnumbering spam messages, which can lead to biased models. Understanding the precise context of a message and identifying nuances of language is another challenge, especially as spam messages become more sophisticated and attempt to mimic legitimate communications. Reducing false positives is also critical and requires careful balancing of the model's decision threshold.
